[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigdata-com/bigdata-cookbook/blob/main/API_Tutorials/CoMentions_API/CoMentions_API_Pepsi.ipynb)

# Competitive Landscape Discovery: PepsiCo via Bigdata.com Co-mentions API

This notebook demonstrates how to use the [Bigdata.com](https://bigdata.com) Co-mentions API to build **entity relationship graphs centered on PepsiCo Inc.**, surfacing competitors, partners, and related entities as they appear across millions of news articles and financial documents.

**What this demonstrates**
- Query the Co-mentions API anchored to PepsiCo to discover which companies, organizations, people, places, and concepts are most frequently mentioned alongside it
- Visualize the results as interactive network graphs with PepsiCo at the center
- Apply a thematic lens to the same query, narrowing co-mentions to a specific topic, to surface a more targeted competitive set

**Two approaches**
1. **Broad entity graph:** no topic filter; discovers the full competitive and contextual landscape around PepsiCo based on raw co-mention frequency across all coverage
2. **Thematic entity graph:** adds a topic query to surface only entities co-mentioned with PepsiCo in the context of a specific strategic theme (e.g. prebiotic and high-fiber product innovation)

**Endpoint documentation:** [Co-mentions API reference](https://docs.bigdata.com/api-reference/co-mentions/connected-entities)


In [ ]:
#!uv pip install -r requirements.txt

In [11]:
import requests
import json
from datetime import datetime, timedelta
from print_helpers import print_comention_results

## Setup & Configuration


In [12]:
# Load credentials from .env file
import os
from dotenv import load_dotenv
load_dotenv()

# Bigdata.com API Configuration
API_BASE_URL = "https://api.bigdata.com"
API_KEY = os.getenv("BIGDATA_API_KEY")
if not API_KEY:
    raise ValueError("Set BIGDATA_API_KEY in .env")

# API Endpoints
SEARCH_ENDPOINT = f"{API_BASE_URL}/v1/search"
COMENTIONS_ENDPOINT = f"{SEARCH_ENDPOINT}/co-mentions/entities"
KG_ENTITIES_ENDPOINT = f"{API_BASE_URL}/v1/knowledge-graph/entities/id"

# Authentication via API key
session = requests.Session()
session.headers.update({"Content-Type": "application/json", "X-API-KEY": API_KEY})
print("✅ API key configured")

✅ API key configured


## Configuration

Set the date range and the focal entity. Here we anchor on **PepsiCo Inc.** (`013528`) and look at period of interest. For this example, we'll use the trailing 12 months.


In [17]:
# Date range for examples
START_DATE = "2025-03-18"
END_DATE = "2026-03-18"
TEXT = ""
ENTITY_ID = "013528"  # Apple Inc

## 1. Broad Entity Graph: No Theme Filter

The first approach runs a co-mentions query anchored to **PepsiCo** with no topic filter. This surfaces every entity (companies, organizations, people, places, products, and concepts) that appears alongside PepsiCo in coverage over the selected date range, ranked purely by co-mention frequency.

This gives an unfiltered view of PepsiCo's full contextual landscape: direct competitors, retail partners, investors, regulators, and any other entities that dominate its news environment. The results are then rendered as interactive network graphs, one per entity category, with PepsiCo at the center.


### Network Graphs by Category

Each graph below places **PepsiCo at the center** and connects it to the top co-mentioned entities in a given category (companies, places, organizations, people, products, concepts). Node size reflects co-mention volume; larger nodes appear more frequently alongside PepsiCo in the corpus. Hover over any node for full entity detail.


In [13]:
from api_helpers import create_comentions_network_graph

# Query co-mentions filtered by entity
comention_query = {
    "query": {
        "text": TEXT,
        "auto_enrich_filters": False,
        "filters": {
            "timestamp": {
                "start": f"{START_DATE}T00:00:00Z",
                "end": f"{END_DATE}T23:59:59Z"
            },
            "entity": {"search_in": "ALL", "any_of": [ENTITY_ID]}
        }
    },
    "limit": 200
}

response = session.post(COMENTIONS_ENDPOINT, json=comention_query)
comentions_data = response.json()

# Display network graphs
if comentions_data.get("results"):
    for category in ["companies", "places", "organizations", "people", "products", "concepts"]:
        entities = comentions_data["results"].get(category, [])
        if entities:
            fig = create_comentions_network_graph(
                session=session,
                kg_entities_endpoint=KG_ENTITIES_ENDPOINT,
                center_name="PepsiCo Inc.",
                center_id=ENTITY_ID,
                connected_entities=entities,
                category_name=category,
                text=TEXT,
                max_nodes=15
            )
            if fig:
                fig.show()

The broad graph reveals PepsiCo's full co-mention universe with no editorial filter. In the next section, we apply a topic query to focus the same graph on a specific strategic theme.


### Corner Cases to Watch

The broad graph can surface entities that rank highly not because of a strategic relationship with PepsiCo, but because of **how they generate coverage**:

- **Media outlets** (e.g. CNN, Reuters, Bloomberg) appear frequently simply because they are the source of the articles; they report on PepsiCo constantly without being meaningfully related to it
- **Rating agencies and financial data providers** (e.g. Moody's, Zacks, S&P) show up when analyst rating changes or earnings estimates are published, inflating their co-mention count without implying any business relationship

In these cases it is advisable to **filter by sector**, restricting results to a specific GICS sector or industry (e.g. Food & Beverage, Consumer Staples), to ensure the graph reflects genuine competitive or thematic peers rather than coverage artifacts. The thematic approach in Section 2 naturally reduces this noise by requiring topical relevance in addition to co-mention frequency.


## 2. Thematic Entity Graph: With a Topic Filter

The second approach adds a **topic query** to the same PepsiCo-anchored co-mentions call. Rather than returning every entity that appears alongside PepsiCo, the API now restricts results to documents where PepsiCo is mentioned in the context of a specific theme: prebiotic and high-fiber product innovation.

This produces a sharper, more actionable competitive set consisting of the companies, organizations, and concepts that share PepsiCo's narrative space on this specific strategic topic. The output is ranked by co-mention frequency (headline count and chunk count), giving two complementary views of prominence within the theme.

The resolved entity IDs can be passed directly into the Search or Volume API (via `entity.any_of`) for deeper signal construction or backtesting.


In [15]:
import pandas as pd

TEXT = "The company is developing prebiotic and high-fiber products"
TOP_N_COMPANIES = 10

# Co-mentions query (topic only, no entity filter)
basket_query = {
    "query": {
        "text": TEXT,
        "auto_enrich_filters": False,
        "filters": {
            "timestamp": {
                "start": f"{START_DATE}T00:00:00Z",
                "end": f"{END_DATE}T23:59:59Z"
            },
            "entity": {"search_in": "ALL", "any_of": [ENTITY_ID]}
        },
        "limit": 500
    },
}

response_basket = session.post(COMENTIONS_ENDPOINT, json=basket_query)
basket_data = response_basket.json()

# API returns merged top-by-chunks and top-by-headlines; we take companies only.
companies_raw = basket_data.get("results", {}).get("companies", [])

# Two baskets: top N companies BY HEADLINE count, top N companies BY CHUNK count.
# Filter to entities that have non-zero count for that metric (merged list can have
# only-headline or only-chunk entities), then take top N. Larger limit helps get more
# companies with both metrics.
with_headlines = [c for c in companies_raw if c.get("total_headlines_count", 0) > 0]
with_chunks = [c for c in companies_raw if c.get("total_chunks_count", 0) > 0]
top_by_headlines = sorted(
    with_headlines,
    key=lambda x: x.get("total_headlines_count", 0),
    reverse=True
)[:TOP_N_COMPANIES]
top_by_chunks = sorted(
    with_chunks,
    key=lambda x: x.get("total_chunks_count", 0),
    reverse=True
)[:TOP_N_COMPANIES]

all_ids = list({e["id"] for e in top_by_headlines + top_by_chunks})
kg_response = session.post(KG_ENTITIES_ENDPOINT, json={"values": all_ids})
resolved = kg_response.json().get("results", {}) if kg_response.status_code == 200 else {}

def make_basket_df(entities, label):
    rows = []
    for i, ent in enumerate(entities, 1):
        name = resolved.get(ent["id"], {}).get("name", ent["id"])
        rows.append({
            "rank": i,
            "entity_id": ent["id"],
            "name": name,
            "headlines": ent.get("total_headlines_count", 0),
            "chunks": ent.get("total_chunks_count", 0),
        })
    return pd.DataFrame(rows)

if not companies_raw:
    print("No companies found for this topic.")
    basket_by_headlines_df = pd.DataFrame()
    basket_by_chunks_df = pd.DataFrame()
else:
    basket_by_headlines_df = make_basket_df(top_by_headlines, "headlines")
    basket_by_chunks_df = make_basket_df(top_by_chunks, "chunks")
    print(f"Thematic basket: top {TOP_N_COMPANIES} companies by headlines and top {TOP_N_COMPANIES} by chunks | \"{TEXT}\" ({START_DATE} to {END_DATE})")
    if len(with_chunks) < TOP_N_COMPANIES:
        print(f"Note: the API returned only {len(with_chunks)} companies with non-zero chunk counts for this topic/period; the by-chunks basket has {len(with_chunks)} rows.")
    print("\n--- Top 10 companies by headline count ---")
    display(basket_by_headlines_df)
    print("\n--- Top 10 companies by chunk count ---")
    display(basket_by_chunks_df)

Thematic basket: top 10 companies by headlines and top 10 by chunks | "The company is developing prebiotic and high-fiber products" (2025-03-18 to 2026-03-18)

--- Top 10 companies by headline count ---


,rank,entity_id,name,headlines,chunks
0,1,013528,PepsiCo Inc.,638,2597
1,2,EEA6B3,Coca-Cola Co.,131,318
2,3,7XVFEZ,Vngr Beverage LLC,82,737
3,4,713810,Walmart Inc.,37,41
4,5,14C7B2,Keurig Dr. Pepper Inc.,22,60
5,6,1F9258,Celsius Holdings Inc.,19,32
6,7,9AF3DC,Kellanova,10,14
7,8,852A1D,Elliott Management Corp.,10,7
8,9,EE46FA,Keurig Inc.,10,0
9,10,3CBA2A,Starbucks Corp.,7,34



--- Top 10 companies by chunk count ---


,rank,entity_id,name,headlines,chunks
0,1,013528,PepsiCo Inc.,638,2597
1,2,7XVFEZ,Vngr Beverage LLC,82,737
2,3,EEA6B3,Coca-Cola Co.,131,318
3,4,14C7B2,Keurig Dr. Pepper Inc.,22,60
4,5,4E7A32,TikTok Inc.,5,59
5,6,5DA676,The Quaker Oats Co.,0,59
6,7,E363D0,PepsiCo Beverages North America,6,46
7,8,AEC277,CNN,0,45
8,9,0157B1,Amazon.com Inc.,3,42
9,10,KXJZ5G,Siete Family Foods Inc.,0,41


In [16]:
# Graphics: bar charts for both baskets (top 10 by headlines, top 10 by chunks)
import plotly.express as px

try:
    has_headlines = basket_by_headlines_df is not None and not basket_by_headlines_df.empty
    has_chunks = basket_by_chunks_df is not None and not basket_by_chunks_df.empty
except NameError:
    has_headlines = has_chunks = False

if has_headlines:
    fig_headlines = px.bar(
        basket_by_headlines_df.sort_values("headlines", ascending=True),
        x="headlines",
        y="name",
        orientation="h",
        title=f'Top 10 companies by headline count<br><sup>"{TEXT}" | {START_DATE} to {END_DATE}</sup>',
        labels={"headlines": "Headline count", "name": "Company"},
        text="headlines",
        text_auto=",.0f",
    )
    fig_headlines.update_layout(
        height=350 + len(basket_by_headlines_df) * 22,
        margin=dict(l=20),
        yaxis=dict(autorange="reversed", tickfont=dict(size=10)),
        showlegend=False,
    )
    fig_headlines.update_traces(textposition="outside")
    fig_headlines.show()

if has_chunks:
    fig_chunks = px.bar(
        basket_by_chunks_df.sort_values("chunks", ascending=True),
        x="chunks",
        y="name",
        orientation="h",
        title=f'Top 10 companies by chunk count<br><sup>"{TEXT}" | {START_DATE} to {END_DATE}</sup>',
        labels={"chunks": "Chunk count", "name": "Company"},
        text="chunks",
        text_auto=",.0f",
    )
    fig_chunks.update_layout(
        height=350 + len(basket_by_chunks_df) * 22,
        margin=dict(l=20),
        yaxis=dict(autorange="reversed", tickfont=dict(size=10)),
        showlegend=False,
    )
    fig_chunks.update_traces(textposition="outside")
    fig_chunks.show()

if not has_headlines and not has_chunks:
    print("Run the cell above first to build the baskets.")